In [ ]:
!pip install --ignore-installed --no-cache-dir libtorrent
import os
import time
import libtorrent as lt

# --- Configuration ---
# The full path to your torrent file
TORRENT_FILE_PATH = '/content/<filename>.torrent'

# The directory where you want to save the downloaded file(s)
DOWNLOADS_PATH = '/content/'

# --- Script Start ---

# 1. Check if the torrent file exists
if not os.path.exists(TORRENT_FILE_PATH):
    print(f"ERROR: Torrent file not found at '{TORRENT_FILE_PATH}'")
    print("Please make sure you have uploaded it to the correct location.")
    raise FileNotFoundError(f"Torrent file not found: {TORRENT_FILE_PATH}")

print("✅ Torrent file found.")
print(f"   File: {os.path.basename(TORRENT_FILE_PATH)}")
print("-" * 50)

# 2. Set up the libtorrent session
ses = lt.session()
ses.listen_on(6881, 6891)
print("✅ Torrent session started.")
print("-" * 50)

# 3. Read the torrent file and add it to the session
print(f"Loading torrent from: {TORRENT_FILE_PATH}")
e = lt.bdecode(open(TORRENT_FILE_PATH, 'rb').read())
params = {
    'save_path': DOWNLOADS_PATH,
    'ti': lt.torrent_info(e)
}
handle = ses.add_torrent(params)
print("✅ Torrent added to session. Starting download...")
print("-" * 50)

# 4. Monitor the download progress
print("Download Status:")
print("(Progress might be slow at first as it connects to peers)")
print("-" * 50)

while not handle.is_seed():
    s = handle.status()
    state_str = ['queued', 'checking', 'downloading metadata', \
                 'downloading', 'finished', 'seeding', 'allocating']

    print('\r' + ' ' * 80, end='')
    print(f"\r{state_str[s.state]}: {s.progress * 100:.2f}% complete " +
          f"(Down: {s.download_rate / 1000:.1f} kB/s | Up: {s.upload_rate / 1000:.1f} kB/s | " +
          f"Peers: {s.num_peers})", end='')

    time.sleep(1)

print("\n" + "-" * 50)
print("🎉 DOWNLOAD COMPLETE! 🎉")
print("-" * 50)

# 5. Print the location of the downloaded file(s)
print("Your file(s) have been downloaded to:")
print(f"   {DOWNLOADS_PATH}")

print("\nContents of the download directory:")
!ls -lh "{DOWNLOADS_PATH}"